# Module 01: ReAct Agents -- Think, Act, Observe

**Estimated time: 60-90 minutes**

---

## Learning Objectives

By the end of this notebook, you will be able to:
- Explain the ReAct loop and why it outperforms simple prompting for structured reasoning tasks
- Read and trace the triage agent implementation
- Understand how structured output extraction works
- Describe each tool available to the agent and what it does

## The ReAct Loop

ReAct = **Re**ason + **Act**. Published by Google DeepMind in 2022, it's now the
foundation of most production agentic AI systems.

The core idea is simple -- give the LLM access to tools, then loop:

```
Input: Security alert
  |
  v
Thought: 'This looks like credential dumping. Let me verify.'
  |
  v
Action: mitre_lookup(query='lsass memory credential access')
  |
  v
Observation: 'T1003.001 -- LSASS Memory. Tactic: Credential Access.'
  |
  v
Thought: 'Confirmed T1003.001. Given persistence earlier, escalating.'
  |
  v
Final Answer: {severity: CRITICAL, techniques: [T1003.001], ...}
```

Each iteration: LLM reasons -> decides what to do -> sees the result -> reasons again.
Repeat until it has enough to give a final answer (or hits the step limit).

**Why this beats a simple prompt:** The agent can gather information it doesn't have upfront.
It's not limited to what you remembered to put in the initial prompt.

In [ ]:
import sys
import os
import inspect

sys.path.insert(0, os.path.abspath('..'))

# Load the triage agent module
from src.agents import triage_agent
print('Triage agent loaded from: ' + str(triage_agent.__file__))

In [ ]:
# Read the system prompt -- every line is a design decision
from src.agents.triage_agent import TRIAGE_PROMPT

print('=== TRIAGE SYSTEM PROMPT ===')
print(TRIAGE_PROMPT.template)

## Anatomy of the System Prompt

Notice the structure:

1. **Role definition** -- 'You are an expert SOC analyst...'
   Sets the behavioral frame. This affects output quality significantly.

2. **Tool descriptions** -- `{tools}` and `{tool_names}`
   LangChain fills these from the tool objects automatically.
   Each tool has a name and description the LLM uses to decide when to call it.

3. **Numbered instructions** -- The step-by-step analysis process
   This is your SOC runbook encoded as AI instructions.
   Think: 'What would a senior analyst do, in what order?'

4. **`{input}`** -- The actual alert content (injected at runtime)

5. **`{agent_scratchpad}`** -- Running log of Thought/Action/Observation
   LangChain maintains this across iterations. The LLM sees its own history.

**Prompt engineering principle:** The clearer and more specific your instructions,
the more consistent and higher-quality the agent's behavior.

In [ ]:
# Inspect the tools available to the agent
from src.tools.mitre_lookup import MitreLookupTool
from src.tools.alert_enrichment import AlertEnrichmentTool
from src.tools.severity_scorer import SeverityScorerTool

tools = [MitreLookupTool(), AlertEnrichmentTool(), SeverityScorerTool()]

for tool in tools:
    print('Tool: ' + tool.name)
    print('  Description: ' + tool.description)
    print()

In [ ]:
# You can call any tool directly -- this is exactly what the agent does mid-reasoning
mitre_tool = MitreLookupTool()

# Try a query similar to what the agent would use for ALERT-2024-001
result = mitre_tool.run('powershell encoded command office process chain')
print('MITRE Lookup Result:')
print(result)

In [ ]:
# After the ReAct loop, the agent outputs free-form text.
# A second LLM call extracts structured JSON reliably.
# Read the extraction prompt pattern:
from src.agents.triage_agent import EXTRACTION_PROMPT

print('=== EXTRACTION PROMPT PATTERN ===')
print(EXTRACTION_PROMPT)
print()
print('Why two LLM calls?')
print('  Triage LLM: temperature=0.1, optimizes for good reasoning')
print('  Extraction LLM: temperature=0.0, optimizes for reliable JSON')
print('  Separating concerns makes each task simpler and more predictable.')

## Exercises

### Beginner
Run `python demo.py` from the repo root. For each of the 5 alerts, note:
- What MITRE techniques did the agent identify?
- What kill chain phase?
- Do you agree with the severity? Why or why not?

### Intermediate
Modify `TRIAGE_PROMPT` in `src/agents/triage_agent.py`:
- Add to the instructions: 'When uncertain between two severity levels, prefer the lower one.'
- Run the demo again. Did any severities change? Check evaluation scores -- did accuracy improve?

### Advanced
Add a new `IPReputationTool` to the agent:
1. Create `src/tools/ip_reputation.py` with a `BaseTool` subclass
2. Hardcode a dict of known-bad IPs with threat intel context
3. Add it to the tools list in `TriageAgent.__init__`
4. Update `TRIAGE_PROMPT` to mention the tool
5. Verify the agent uses it when processing ALERT-2024-002 (contains a suspicious IP)

---

## Module 01 Complete

You've:
- Traced the full ReAct loop from prompt to final decision
- Inspected all three agent tools
- Understood the two-LLM pattern (generation + extraction)

**Next: [Module 02 -- LLM Providers](02_llm_providers.ipynb)**
Run the real agent for $0 using Ollama or Groq.